### 0. Default Setting

#### 1) Import Library

In [1]:
import tensorflow as tf
import numpy as np 
import pandas as pd 
import os
from keras.preprocessing.text import Tokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from sklearn import metrics
from keras_preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.layers import SpatialDropout1D
from tensorflow.keras.layers import Conv1D, Bidirectional, LSTM, Dense, Input, Dropout

2023-09-25 14:15:04.584352: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-09-25 14:15:05.275363: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


#### 2) Set path

In [2]:
default_path = os.getcwd()
data_path = os.path.join(default_path, '../data')
base_model = os.path.join(default_path, '../base-model')
model_path = os.path.join(default_path, '../models')
config_path = os.path.join(default_path, '../config')
log_path = os.path.join(default_path, '../log')
config_file = "bert-base.json"

#### 3) Load Data

In [3]:
dsm_train = pd.read_csv(os.path.join(data_path, 'emodep_train.csv'))
dsm_val = pd.read_csv(os.path.join(data_path, 'emodep_val.csv'))
dsm_test = pd.read_csv(os.path.join(data_path, 'emodep_test.csv'))

In [4]:
dsm_train = dsm_train[['author', 'text', 'type', 'criteria']]
dsm_val = dsm_val[['author', 'text', 'type', 'criteria']]
dsm_test = dsm_test[['author', 'text', 'type', 'criteria']]

dsm_train.columns = ['author', 'text', 'type', 'label']
dsm_val.columns = ['author', 'text', 'type', 'label']
dsm_test.columns = ['author', 'text', 'type', 'label']

### 2. Pre-process data

In [5]:
tfidf_vector = TfidfVectorizer(lowercase=True, 
                         stop_words='english', 
                         analyzer='word',  # tokens should be words. we can also use char for character tokens
                         max_features=1000, # maximum vocabulary size to restrict too many features
                         min_df = 5)

dsm_train_corpus = tfidf_vector.fit_transform(dsm_train.text)

In [6]:
dsm_val_corpus = tfidf_vector.transform(dsm_val.text) 
dsm_test_corpus = tfidf_vector.transform(dsm_test.text)

In [7]:
dsm_train_y = pd.get_dummies(dsm_train['label']).values
dsm_val_y = pd.get_dummies(dsm_val['label']).values
dsm_test_y = pd.get_dummies(dsm_test['label']).values

### 3. Modeling

#### 1) Classification

In [48]:
(dsm_train_corpus.shape[1], )

(1000,)

In [ ]:
expected ndim=3, found ndim=2. Full shape received: (None, 1000)

In [52]:
dsm_train_corpus = dsm_train_corpus.reshape(-1, 1, 1000)
dsm_val_corpus = dsm_val_corpus.reshape(-1, 1, 1000)
dsm_test_corpus = dsm_test_corpus.reshape(-1, 1, 1000)

ValueError: matrix shape must be two-dimensional

In [50]:
dsm_train_corpus.shape

(156492, 1000)

In [38]:
model = Sequential()
model.add(Bidirectional(LSTM(10, return_sequences=True), input_shape=(dsm_train_corpus.shape[1], )))
# model.add(Bidirectional(LSTM(10)))
model.add(Dense(9,activation='softmax'))
model.summary()

ValueError: Input 0 of layer "bidirectional_12" is incompatible with the layer: expected ndim=3, found ndim=2. Full shape received: (None, 1000)

In [29]:
model.compile(loss='categorical_crossentropy',metrics=['accuracy',])

In [26]:
np.shape(dsm_train_corpus.toarray()), np.shape(dsm_train_corpus.toarray()[1])

((156492, 1000), (1000,))

In [30]:
model.fit(dsm_train_corpus.toarray(), dsm_train_y, epochs=5, batch_size=8, verbose=1)

Epoch 1/5


ValueError: in user code:

    File "/home/pbl1/anaconda3/envs/tensor/lib/python3.8/site-packages/keras/src/engine/training.py", line 1338, in train_function  *
        return step_function(self, iterator)
    File "/home/pbl1/anaconda3/envs/tensor/lib/python3.8/site-packages/keras/src/engine/training.py", line 1322, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "/home/pbl1/anaconda3/envs/tensor/lib/python3.8/site-packages/keras/src/engine/training.py", line 1303, in run_step  **
        outputs = model.train_step(data)
    File "/home/pbl1/anaconda3/envs/tensor/lib/python3.8/site-packages/keras/src/engine/training.py", line 1080, in train_step
        y_pred = self(x, training=True)
    File "/home/pbl1/anaconda3/envs/tensor/lib/python3.8/site-packages/keras/src/utils/traceback_utils.py", line 70, in error_handler
        raise e.with_traceback(filtered_tb) from None
    File "/home/pbl1/anaconda3/envs/tensor/lib/python3.8/site-p

In [40]:
y_predict = model.predict(dsm_test_corpus.toarray())

1359/1359 [==============================] - 1s 819us/step


In [44]:
dsm_pred = []
for val in y_predict:
    dsm_pred.append(np.argmax(val))
    
dsm_pred[0:5]

[1, 0, 8, 1, 8]

In [47]:
pd.DataFrame(dsm_pred, columns=['dnn']).to_csv(os.path.join(data_path, 'dsm_dnn_pred.csv'), index=False)

In [45]:
from torchmetrics.classification import F1Score, MulticlassPrecision, MulticlassRecall, MulticlassSpecificity

f1 = F1Score(task="multiclass", num_classes=10)
precision = MulticlassPrecision(num_classes=10)
recall = MulticlassRecall(num_classes=10)
specificity = MulticlassSpecificity(num_classes=10)

print(f'dsm precision: {precision(torch.Tensor(dsm_pred), torch.Tensor(dsm_true))}')
print(f'dsm recall: {recall(torch.Tensor(dsm_pred), torch.Tensor(dsm_true))}')
print(f'dsm specificity: {specificity(torch.Tensor(dsm_pred), torch.Tensor(dsm_true))}')
print(f'dsm f1-score: {f1(torch.Tensor(dsm_pred), torch.Tensor(dsm_true))}')

NameError: name 'torch' is not defined

#### 2) Regression

In [97]:
model = Sequential()
# model.add(Bidirectional(LSTM(10, return_sequences=True), input_shape=train_corpus.toarray().shape))
model.add(Dense(256,input_shape = (bws_train_corpus.shape[1],), activation = 'relu'))
model.add(Dense(1,activation='relu'))
model.summary()

Model: "sequential_19"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_27 (Dense)            (None, 256)               54784     
                                                                 
 dense_28 (Dense)            (None, 1)                 257       
                                                                 
Total params: 55,041
Trainable params: 55,041
Non-trainable params: 0
_________________________________________________________________


In [98]:
model.compile(loss='mse',metrics=['accuracy',])

In [100]:
model.fit(bws_train_corpus.toarray(), bws_train_y, epochs=5, batch_size=8, verbose=1)

Epoch 1/5
144/144 [==============================] - 0s 1ms/step - loss: 49.1943 - accuracy: 0.0477
Epoch 2/5
144/144 [==============================] - 0s 1ms/step - loss: 17.3325 - accuracy: 0.0095
Epoch 3/5
144/144 [==============================] - 0s 886us/step - loss: 14.1156 - accuracy: 0.0191
Epoch 4/5
144/144 [==============================] - 0s 833us/step - loss: 12.2428 - accuracy: 0.0312
Epoch 5/5
144/144 [==============================] - 0s 2ms/step - loss: 10.8793 - accuracy: 0.0477


In [38]:
y_predict = model.predict(bws_test_corpus.toarray())

NameError: name 'bws_test_corpus' is not defined

In [111]:
y_pred = [p[0] for p in y_predict]
y_pred[:3]

[8.538281, 8.579833, 10.127542]